# voice-rs — huấn luyện adapter trên Colab (CPU)

Luồng: clone repo → **tải model** (`scripts/fetch_assets.sh`) → **tải binary đã build sẵn**
(`scripts/fetch_bin.sh`) → chạy `scripts/run_training.sh` (embed → train → validate) → tải kết quả về.

**Không cần cài Rust và không biên dịch gì** — Colab chỉ tải bundle model + binary từ
GitHub Release rồi chạy. Cũng không cần PyTorch.

Chạy trong **Runtime → Change runtime type → CPU** (chưa hỗ trợ GPU).

⚠️ chi phí: Colab free ~2 core. `SEG=86` (1 s) ≈ 1.5–2 h cho 250 bước.
Muốn cả câu (589 frame) thì đặt `SEG=589` — chậm hơn ~7×.


## 1. Clone repo


In [ ]:
%%bash
set -e
cd /content
[ -d voice-rs ] || git clone https://github.com/hungpham10/voice-rs.git
cd voice-rs && git log --oneline -1


## 2. Tải model từ Release `assets-v1`
`voice-assets.tar.gz` = ContentVec ONNX + `vits_ljs_dec` (VITS generator đã export).


In [ ]:
%%bash
set -e
cd /content/voice-rs
./scripts/fetch_assets.sh assets-v1


## 3. Tải binary train/validate (không build)
`voice`, `verify_dec` (Linux x86_64, do cargo-dist đóng gói).


In [ ]:
%%bash
set -e
cd /content/voice-rs
./scripts/fetch_bin.sh


## 4. Validation + training
`SEG` = frame latent mỗi mẫu; `BATCH`/`STEPS`/`LR` tuỳ chỉnh qua biến môi trường.


In [ ]:
%%bash
set -e
cd /content/voice-rs
SEG=86 STEPS=250 LR=1e-3 ./scripts/run_training.sh 2>&1 | tee /content/train.log


## 5. Tải kết quả về máy


In [ ]:
from google.colab import files
files.download("/content/voice-rs/test_output.wav")
files.download("/content/voice-rs/sources/features/adapter.bpk")
